# 03 · Validación fuera de muestra

## Paso 6 · Walk-forward

Si se eligen parámetros mirando todo el historial y se reporta ese mismo historial, el resultado está inflado: se elige lo que funcionó por casualidad. Aquí los parámetros se optimizan **solo con el pasado de cada ventana**.

**Esquema (ventana móvil):** entrenar con 5 años → elegir los parámetros con mejor Sharpe → aplicarlos al **año siguiente**, que no se vio → avanzar un año y repetir. Los años de prueba concatenados forman la curva fuera de muestra: 16 años, de 2010 a 2025.

### 6.1 Rejillas de parámetros, declaradas de antemano (32 intentos en total)

| Estrategia | Valores probados | Combinaciones |
|---|---|---|
| Cruce de medias | fast: 20, 50, 100; slow: 100, 150, 200, 250 (fast < slow) | 11 |
| Momentum | lookback: 126, 189, 252; skip: 0, 21 | 6 |
| Reversión a la media | window: 10, 20, 40; entry_z: −0.5, −1.0, −1.5 | 9 |
| Volatilidad objetivo | target: 0.08, 0.10, 0.15; window: 21, 63 | 6 |

Simplificación declarada: al cambiar de parámetros entre ventanas no se cobra el costo de la transición.

In [1]:
import sys; sys.path.append("..")
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data import download_prices
from src.signals import STRATEGIES
from src.backtest import run_backtest
from src.metrics import summarize, sharpe
from src.validation import GRIDS, param_combos, walk_forward
sns.set_theme(style="whitegrid")

px = download_prices(cache="../data/raw/prices.parquet")
print({k: len(list(param_combos(g))) for k, g in GRIDS.items()},
      "| total:", sum(len(list(param_combos(g))) for g in GRIDS.values()))

{'sma_cross': 11, 'momentum': 6, 'mean_reversion': 9, 'vol_target': 6} | total: 32


### 6.3 Walk-forward y comparación de tres versiones sobre el mismo periodo

In [2]:
t0 = time.time()
oos, chosen = {}, {}
for name in GRIDS:
    oos[name], chosen[name] = walk_forward(px, STRATEGIES[name], GRIDS[name])
print(f"walk-forward completo en {time.time() - t0:.0f} s")

oos_start = oos["sma_cross"].index[0]
bench = run_backtest(px, STRATEGIES["buy_and_hold"](px), start=oos_start)
print("Periodo fuera de muestra:", oos_start.date(), "a", bench.index[-1].date())
assert all(r.index.equals(bench.index) for r in oos.values())   # mismas fechas para todos

rows = [summarize(bench, "buy_and_hold")]
for n in GRIDS:
    fixed = run_backtest(px, STRATEGIES[n](px), start=oos_start)
    rows += [summarize(fixed, f"{n} (parámetros fijos)"),
             summarize(oos[n], f"{n} (walk-forward)")]
compare = pd.DataFrame(rows).set_index("strategy").round(3)
compare

walk-forward completo en 1 s
Periodo fuera de muestra: 2010-01-04 a 2025-12-30


,cagr,vol,sharpe,sortino,max_dd,calmar,turnover_yr,cost_drag_yr
strategy,,,,,,,,
buy_and_hold,0.106,0.116,0.924,1.324,-0.261,0.405,0.000,0.000
sma_cross (parámetros fijos),0.072,0.092,0.806,1.126,-0.185,0.390,1.366,0.001
sma_cross (walk-forward),0.072,0.089,0.822,1.145,-0.185,0.389,1.979,0.001
momentum (parámetros fijos),0.081,0.096,0.864,1.211,-0.185,0.440,5.475,0.003
momentum (walk-forward),0.060,0.084,0.732,1.015,-0.147,0.409,6.828,0.003
mean_reversion (parámetros fijos),0.022,0.074,0.334,0.482,-0.145,0.153,27.023,0.014
mean_reversion (walk-forward),0.021,0.075,0.319,0.457,-0.153,0.140,20.333,0.010
vol_target (parámetros fijos),0.062,0.065,0.952,1.341,-0.143,0.431,4.862,0.002
vol_target (walk-forward),0.059,0.063,0.948,1.344,-0.124,0.479,2.923,0.001


### Prueba: la validación no mira el futuro
Se alteran al azar todos los precios desde 2016. Si el walk-forward fuera honesto, los años de prueba 2010-2015 deben quedar **idénticos**, tanto los retornos como los parámetros elegidos.

In [3]:
px_alt = px.copy()
tail = px_alt.index >= "2016-01-01"
px_alt.loc[tail] = px_alt.loc[tail].values * np.random.default_rng(1).uniform(0.5, 1.5, size=(tail.sum(), px.shape[1]))
for name in GRIDS:
    alt_r, alt_c = walk_forward(px_alt, STRATEGIES[name], GRIDS[name])
    pd.testing.assert_series_equal(oos[name].loc[:"2015", "net"], alt_r.loc[:"2015", "net"])
    pd.testing.assert_frame_equal(chosen[name].loc[:2015], alt_c.loc[:2015])
    assert not oos[name].loc["2016":, "net"].equals(alt_r.loc["2016":, "net"])   # el futuro sí cambia
    print(name, "ok: 2010-2015 idénticos; 2016+ cambian")

sma_cross ok: 2010-2015 idénticos; 2016+ cambian


momentum ok: 2010-2015 idénticos; 2016+ cambian


mean_reversion ok: 2010-2015 idénticos; 2016+ cambian


vol_target ok: 2010-2015 idénticos; 2016+ cambian


### 6.4 Degradación: Sharpe en entrenamiento vs. fuera de muestra

In [4]:
deg = pd.DataFrame({n: {"sharpe_entrenamiento_medio": chosen[n]["train_sharpe"].mean(),
                         "sharpe_fuera_de_muestra": sharpe(oos[n]["net"])} for n in GRIDS}).T
deg["degradacion"] = deg["sharpe_fuera_de_muestra"] - deg["sharpe_entrenamiento_medio"]
deg.round(3)

,sharpe_entrenamiento_medio,sharpe_fuera_de_muestra,degradacion
sma_cross,0.919,0.822,-0.097
momentum,0.879,0.732,-0.147
mean_reversion,0.503,0.319,-0.183
vol_target,0.943,0.948,0.006


### 6.5 Estabilidad de los parámetros elegidos año con año

In [5]:
for n in GRIDS:
    params = [c for c in chosen[n].columns if c != "train_sharpe"]
    combos = chosen[n][params].astype(str).agg("/".join, axis=1)
    print(f"\n{n}: {combos.nunique()} combinaciones distintas en 16 años; "
          f"cambia de un año a otro {int((combos != combos.shift()).iloc[1:].sum())} veces")
    print(combos.value_counts().to_string())


sma_cross: 4 combinaciones distintas en 16 años; cambia de un año a otro 4 veces
50/100     5
50/150     5
100/250    4
20/100     2

momentum: 6 combinaciones distintas en 16 años; cambia de un año a otro 8 veces
126/0     4
252/0     3
252/21    3
189/0     3
189/21    2
126/21    1

mean_reversion: 5 combinaciones distintas en 16 años; cambia de un año a otro 5 veces
40/-0.5    10
10/-1.5     2
40/-1.5     2
40/-1.0     1
20/-1.5     1

vol_target: 5 combinaciones distintas en 16 años; cambia de un año a otro 8 veces
0.08/21    5
0.15/63    4
0.08/63    3
0.1/63     2
0.1/21     2


In [6]:
chosen["sma_cross"].round(2)

,fast,slow,train_sharpe
test_year,,,
2010,50,100,1.00
2011,50,100,0.99
2012,50,100,1.00
2013,50,100,0.87
2014,50,100,1.10
2015,50,150,0.92
2016,50,150,0.72
2017,100,250,0.72
2018,100,250,1.13


### 6.6 Desempeño por año fuera de muestra

In [7]:
yearly = pd.DataFrame({n: r["net"].groupby(r.index.year).apply(sharpe) for n, r in oos.items()})
yearly["buy_and_hold"] = bench["net"].groupby(bench.index.year).apply(sharpe)

plt.figure(figsize=(10, 4))
sns.heatmap(yearly.T, annot=True, fmt=".1f", cmap="RdYlGn", center=0)
plt.title("Sharpe por año, fuera de muestra (walk-forward)")
plt.tight_layout()
plt.savefig("../reports/figures/07_yearly_sharpe_oos.png", dpi=150)

eq = pd.DataFrame({n: (1 + r["net"]).cumprod() for n, r in oos.items()})
eq["buy_and_hold"] = bench["equity"]
eq.plot(logy=True, figsize=(9, 5), title="Curvas fuera de muestra (walk-forward)")
plt.tight_layout()
plt.savefig("../reports/figures/08_oos_equity.png", dpi=150)

beats = (yearly.drop(columns="buy_and_hold").gt(yearly["buy_and_hold"], axis=0)).sum()
print("Años (de 16) en que cada estrategia supera el Sharpe anual de buy & hold:")
print(beats.to_string())
print("\nCapital final fuera de muestra:"); print(eq.iloc[-1].round(2).sort_values(ascending=False).to_string())

Años (de 16) en que cada estrategia supera el Sharpe anual de buy & hold:
sma_cross         5
momentum          7
mean_reversion    4
vol_target        6

Capital final fuera de muestra:
buy_and_hold      4.96
sma_cross         3.03
momentum          2.53
vol_target        2.51
mean_reversion    1.40


### Conclusiones del Paso 6 (fuera de muestra, 2010-01-04 a 2025-12-30, 5 bps, delay = 1)

- **La validación no mira el futuro (comprobado):** al alterar al azar todos los precios desde 2016, los resultados y los parámetros elegidos para 2010-2015 no cambian en nada.
- **Ninguna estrategia supera claramente a buy & hold fuera de muestra.** Buy & hold tiene Sharpe **0.92** en este periodo. Solo la volatilidad objetivo lo iguala (**0.95** con walk-forward); cruce de medias 0.82, momentum 0.73 y reversión a la media 0.32 quedan por debajo. La diferencia de la volatilidad objetivo (+0.02) es irrelevante frente al error estándar del Sharpe con 16 años (≈ 0.25).
- **Optimizar no ayudó.** Con walk-forward el Sharpe fuera de muestra es igual o peor que con los parámetros fijos de la literatura: cruce de medias 0.82 vs. 0.81, volatilidad objetivo 0.95 vs. 0.95, reversión 0.32 vs. 0.33 y **momentum 0.73 vs. 0.86**. Elegir parámetros con el pasado no añadió ventaja.
- **Degradación (entrenamiento → prueba):** momentum −0.15, reversión a la media −0.18, cruce de medias −0.10. Es la medida honesta del sobreajuste: lo que parecía bueno al elegir se debilita en datos nuevos. La volatilidad objetivo no se degrada (+0.01).
- **Parámetros inestables:** momentum y volatilidad objetivo cambian de parámetros 8 veces en 16 años y prueban 5-6 combinaciones distintas: la "mejor" combinación es casi ruido. El cruce de medias es más estable (4 cambios). La reversión a la media prefiere casi siempre la versión que menos opera (40 días, z = −0.5).
- **Año por año:** ninguna estrategia supera el Sharpe anual de buy & hold en más de 7 de 16 años. En **2022** (acciones y bonos cayendo juntos) las estrategias de tendencia fueron **peores** que buy & hold (cruce de medias −2.4 y momentum −2.2 frente a −1.3). 2017 y 2019 fueron buenos para todas.
- **Lo que sí se mantiene:** menor riesgo. Caída máxima fuera de muestra de −12 % a −19 % frente a −26 % de buy & hold, y Calmar de la volatilidad objetivo 0.48 vs. 0.41. A cambio de menos retorno ($1 → 2.5-3.0 frente a 4.96).

**Veredicto preliminar:** con datos que no vieron, estas reglas simples no generan ventaja ajustada por riesgo sobre buy & hold; como mucho la igualan con menos caídas. El Paso 7 lo pondrá a prueba formalmente (intervalos de confianza, Deflated Sharpe y placebo de timing).
